## Mount Google Drive


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Install Libraries


In [2]:
!pip install -q numpy pandas matplotlib scikit-learn statsforecast sktime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.8/501.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 2.6 MB/s eta 0:00:00


## Imports


In [3]:
import os
import json
import gc
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from statsforecast.models import AutoETS
from sktime.performance_metrics.forecasting import MeanAbsoluteScaledError


## JSON Data File


In [4]:
CONFIG_PATH = '/content/drive/MyDrive/Models parameters JSON files/seasonalnaive.json'

## Load Configuration


In [5]:
with open(CONFIG_PATH, 'r', encoding='utf-8') as file:
    CONFIG = json.load(file)

## Read Dataset List


In [6]:
DATASETS = CONFIG.get('datasets', [])

## Global Settings


In [7]:
# These settings are defined in the code because the JSON contains datasets only.
TRAIN_RATIO = 0.80
SEASON_LENGTH = 12
OUTPUT_DIR = '/content/drive/MyDrive/results/json/AutoETS'
SHOW_PLOTS = True

total_models = sum(len(dataset.get('targets', [])) for dataset in DATASETS)
completed_models = 0
all_results = []

## Create Output Directory


In [8]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Print Configuration


In [9]:
print('AUTOETS FORECAST CONFIGURATION')
print('Data file:', CONFIG_PATH)
print('Datasets:', len(DATASETS))
print('Targets:', total_models)
print('Train ratio:', TRAIN_RATIO)
print('Season length:', SEASON_LENGTH)
print('Output directory:', OUTPUT_DIR)
print('Show plots:', SHOW_PLOTS)

AUTOETS FORECAST CONFIGURATION
Data file: /content/drive/MyDrive/Models parameters JSON files/seasonalnaive.json
Datasets: 6
Targets: 26
Train ratio: 0.8
Season length: 12
Output directory: /content/drive/MyDrive/results/json/AutoETS
Show plots: True


## Metrics


In [10]:
def scale_values(y_train, y_true, y_pred):
    y_train = np.asarray(y_train, dtype=np.float64).reshape(-1, 1)
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1, 1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1, 1)

    scaler = MinMaxScaler()
    scaler.fit(y_train)
    y_true_scaled = scaler.transform(y_true)
    y_pred_scaled = scaler.transform(y_pred)

    return y_true_scaled, y_pred_scaled


##  Save Complete Time-Series JSON


In [11]:
def save_timeseries_full(
    log_path,
    model_name,
    full_dates,
    full_values,
    train_dates,
    train_values,
    test_dates,
    test_values,
    forecast_dates,
    forecast_values,
    metrics
):
    # Read previous results from the JSON file.
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8') as file:
            log_data = json.load(file)
    else:
        log_data = []

    # Create the result for this model.
    entry = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model': model_name,
        'full_series': {
            'dates': [str(date) for date in full_dates],
            'values': [float(value) for value in full_values]
        },
        'train_series': {
            'dates': [str(date) for date in train_dates],
            'values': [float(value) for value in train_values]
        },
        'test_series': {
            'dates': [str(date) for date in test_dates],
            'values': [float(value) for value in test_values]
        },
        'forecast': {
            'dates': [str(date) for date in forecast_dates],
            'prediction': [float(value) for value in forecast_values]
        },
        'metrics': {
            'mae': float(metrics['mae']),
            'mse': float(metrics['mse']),
            'rmse': float(metrics['rmse']),
            'mae_scaled': float(metrics['mae_scaled']),
            'mse_scaled': float(metrics['mse_scaled']),
            'rmse_scaled': float(metrics['rmse_scaled']),
            'mase': None if np.isnan(metrics['mase']) else float(metrics['mase'])
        }
    }

    # Add the new result and save it.
    log_data.append(entry)

    with open(log_path, 'w', encoding='utf-8') as file:
        json.dump(log_data, file, indent=4, ensure_ascii=False)


## Prepare one target column


In [12]:
# Cell 1: Prepare one target column
def prepare_target(data, dataset, target):
    dataset_name = dataset['dataset_name']
    time_column = dataset['time_column']
    target_column = target['column']

    target_data = data[[time_column, target_column]].copy()
    target_data[target_column] = pd.to_numeric(target_data[target_column])

    dates = target_data[time_column]
    values = target_data[target_column].to_numpy(dtype=float)

    return dataset_name, time_column, target_column, dates, values

## Split data into training and test parts

In [13]:
# Cell 2: Split data into training and test parts
def split_data(dates, values):
    split_index = int(len(values) * TRAIN_RATIO)

    train_values = values[:split_index]
    test_values = values[split_index:]
    train_dates = dates.iloc[:split_index]
    test_dates = dates.iloc[split_index:]

    return train_dates, train_values, test_dates, test_values

##Create an AutoETS forecast

In [14]:
# Cell 3: Create an AutoETS forecast
def create_forecast(train_values, test_values):
    model = AutoETS(season_length=SEASON_LENGTH)
    model.fit(y=train_values)
    print('Selected ETS components and parameters:', model.model_['components'], model.model_['par'][:4])
    forecast = model.predict(h=len(test_values))

    return np.asarray(forecast['mean'], dtype=float)

## Calculate forecast errors

In [15]:
# Cell 4: Calculate forecast errors
def calculate_metrics(train_values, test_values, predictions):
    test_array = np.asarray(test_values, dtype=np.float64)
    prediction_array = np.asarray(predictions, dtype=np.float64)

    mase = MeanAbsoluteScaledError()(
        test_array, prediction_array, y_train=train_values
    )

    # Unscaled errors use the original values.
    mae = float(mean_absolute_error(test_array, prediction_array))
    mse = float(mean_squared_error(test_array, prediction_array))
    rmse = float(np.sqrt(mse))

    # Scaled errors use a MinMaxScaler fitted only on the training values.
    test_scaled, predictions_scaled = scale_values(
        train_values, test_array, prediction_array
    )
    scaled_mae = float(mean_absolute_error(test_scaled, predictions_scaled))
    scaled_mse = float(mean_squared_error(test_scaled, predictions_scaled))
    scaled_rmse = float(np.sqrt(scaled_mse))

    return {
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'mase': float(mase),
        'mae_scaled': scaled_mae,
        'mse_scaled': scaled_mse,
        'rmse_scaled': scaled_rmse,
    }


## Draw the forecast chart

In [16]:
# Cell 5: Draw the forecast chart
def show_forecast(train_dates, train_values, test_dates, test_values, predictions,
                  dataset_name, time_column, target_column):
    if SHOW_PLOTS:
        plt.figure(figsize=(16, 6))
        plt.plot(train_dates, train_values, label='Training data')
        plt.plot(test_dates, test_values, label='Actual values')
        plt.plot(test_dates, predictions, '--', label='AutoETS forecast')
        plt.title(f'{dataset_name} - {target_column}')
        plt.xlabel(time_column)
        plt.ylabel(target_column)
        plt.legend()
        plt.grid()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## Save one result as JSON

In [17]:
# Cell 6: Save one result as JSON
def save_result(dataset_name, target_column, target, dates, values,
                train_dates, train_values, test_dates, test_values,
                predictions, metrics):
    output_file = target['output_file']
    log_path = os.path.join(OUTPUT_DIR, output_file)
    model_name = f'AutoETS_s{SEASON_LENGTH}_{dataset_name}_{target_column}'

    save_timeseries_full(
        log_path, model_name, dates, values,
        train_dates, train_values, test_dates, test_values,
        test_dates, predictions, metrics
    )

    all_results.append({
        'dataset': dataset_name,
        'target': target_column,
        'model': model_name,
        'mae': metrics['mae'],
        'mse': metrics['mse'],
        'rmse': metrics['rmse'],
        'mae_scaled': metrics['mae_scaled'],
        'mse_scaled': metrics['mse_scaled'],
        'rmse_scaled': metrics['rmse_scaled'],
        'mase': metrics['mase']
    })

    print('JSON saved:', log_path)


## Run all steps for one target column

In [18]:
# Cell 7: Run all steps for one target column
def run_target(data, dataset, target, model_number):
    dataset_name, time_column, target_column, dates, values = prepare_target(
        data, dataset, target
    )

    train_dates, train_values, test_dates, test_values = split_data(dates, values)
    predictions = create_forecast(train_values, test_values)
    metrics = calculate_metrics(train_values, test_values, predictions)

    print(f'\nModel {model_number}/{total_models}: {dataset_name} - {target_column}')
    print('MAE (scaled):', round(metrics['mae_scaled'], 4))
    print('RMSE (scaled):', round(metrics['rmse_scaled'], 4))
    print('MASE:', round(metrics['mase'], 4))

    show_forecast(
        train_dates, train_values, test_dates, test_values, predictions,
        dataset_name, time_column, target_column
    )

    save_result(
        dataset_name, target_column, target, dates, values,
        train_dates, train_values, test_dates, test_values,
        predictions, metrics
    )


## Run all datasets and target columns

In [19]:
# Cell 8: Run all datasets and target columns
all_results = []
completed_models = 0

for dataset in DATASETS:
    dataset_name = dataset['dataset_name']
    file_path = dataset['file_path']
    time_column = dataset['time_column']

    data = pd.read_csv(file_path)
    data[time_column] = pd.to_datetime(data[time_column])
    data = data.sort_values(time_column).reset_index(drop=True)

    print('\n' + '=' * 60)
    print('Dataset:', dataset_name)

    for target in dataset['targets']:
        completed_models += 1
        run_target(data, dataset, target, completed_models)

Output hidden; open in https://colab.research.google.com to view.

##  Final Summary


In [22]:
print('\n\n')
print('=' * 110)
print('ALL DATASETS / TARGETS COMPLETED')
print('=' * 110)
if len(all_results) > 0:
    summary_df = pd.DataFrame(all_results)
    print(summary_df.to_string(index=False))
else:
    print('⚠️ No target was successfully completed.')




ALL DATASETS / TARGETS COMPLETED
dataset         target                            model           mae          mse          rmse  mae_scaled  mse_scaled  rmse_scaled      mase
   WB90             wt              AutoETS_s12_WB90_wt      0.267797 1.267514e-01      0.356022    0.098455    0.017132     0.130890  1.992577
   WB90            toc             AutoETS_s12_WB90_toc      0.047093 3.454271e-03      0.058773    0.100198    0.015637     0.125049  0.550285
   WB90    temperature     AutoETS_s12_WB90_temperature      2.695467 1.068848e+01      3.269324    0.284032    0.118682     0.344502  4.761872
   WB90             ph              AutoETS_s12_WB90_ph      0.059422 6.790339e-03      0.082404    0.185692    0.066312     0.257511  0.830096
   WB90       humidity        AutoETS_s12_WB90_humidity      9.861109 1.420185e+02     11.917154    0.209811    0.064291     0.253556  4.171249
   WB90             do              AutoETS_s12_WB90_do      0.080069 1.257093e-02      0.112120    

## Information


In [23]:
print('\n')
print('=' * 90)
print('PROCESS FINISHED')
print('=' * 90)
print('Datasets requested:', len(DATASETS))
print('Targets requested:', total_models)
print('Targets completed:', len(all_results))
print('Output directory:', OUTPUT_DIR)
if len(all_results) != total_models:
    print('\n⚠️ WARNING:')
    print('Not all requested targets completed successfully.')
    print('Requested:', total_models)
    print('Completed:', len(all_results))
else:
    print('\n✔️ ALL REQUESTED TARGETS COMPLETED.')



PROCESS FINISHED
Datasets requested: 6
Targets requested: 26
Targets completed: 26
Output directory: /content/drive/MyDrive/results/json/AutoETS

✔️ ALL REQUESTED TARGETS COMPLETED.
